# Pairwise K-Means + Agglomerative clustering (BMI/Glucose/Age)

Creates three pair datasets:
- `BMI vs glucose`
- `BMI vs age`
- `glucose vs age`

Then runs K-Means and Agglomerative clustering on each pair separately (Agglomerative includes a dendrogram).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, pairwise_distances_argmin_min
from sklearn.preprocessing import StandardScaler

from scipy.cluster.hierarchy import dendrogram, linkage

%matplotlib inline

In [2]:
data = pd.read_csv('combined_cleaned.csv')
print('Data shape:', data.shape)

required_cols = ['age', 'glucose', 'bmi']
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f"combined_cleaned.csv missing columns: {missing}")
print('Using columns:', required_cols)

Data shape: (194805, 14)
Using columns: ['age', 'glucose', 'bmi']


In [3]:
def run_pairwise_clustering(df, x_col, y_col, pair_name):
    # pair_name examples: 'bmi_glucose', 'bmi_age', 'glucose_age'
    pair_df = df[[x_col, y_col]].copy()

    pair_csv = f'combined_cleaned_{pair_name}.csv'
    pair_df.to_csv(pair_csv, index=False)
    print(f'\n=== {pair_name}: saved {pair_csv} ===')

    # Preprocess for clustering (impute + scale)
    imputer = SimpleImputer(strategy='median')
    X_pair = imputer.fit_transform(pair_df)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_pair)

    # Choose k for this pair (K-Means silhouette on scaled 2D space)
    k_values = range(2, 11)
    sil_scores = []
    silhouette_sample_size = min(10000, X_scaled.shape[0])

    for k in k_values:
        km = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = km.fit_predict(X_scaled)
        sil_scores.append(
            silhouette_score(
                X_scaled,
                labels,
                sample_size=silhouette_sample_size,
                random_state=42,
            )
        )

    best_k = int(list(k_values)[int(np.argmax(sil_scores))])
    print(f'Best k by silhouette for {pair_name}:', best_k)

    # K-Means fit
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
    clusters_kmeans = kmeans.fit_predict(X_scaled)

    out_k = df[[x_col, y_col]].copy()
    out_k[f'kmeans_cluster_{pair_name}'] = clusters_kmeans
    out_k_path = f'combined_cleaned_{pair_name}_kmeans.csv'
    out_k.to_csv(out_k_path, index=False)
    print('K-Means cluster counts:')
    print(out_k[f'kmeans_cluster_{pair_name}'].value_counts().sort_index())

    # K-Means scatter save
    plt.figure(figsize=(7, 5))
    plt.scatter(out_k[x_col], out_k[y_col], c=clusters_kmeans, s=5, cmap='tab10', alpha=0.6)
    plt.title(f'K-Means clusters ({pair_name}), k={best_k}')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.savefig(f'kmeans_scatter_{pair_name}.png', dpi=200, bbox_inches='tight')
    plt.close()

    # Agglomerative on a sample; label full using nearest-centroid approximation
    sample_size = min(10000, X_scaled.shape[0])
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(X_scaled.shape[0], size=sample_size, replace=False)
    X_sample = X_scaled[sample_idx]

    agg = AgglomerativeClustering(n_clusters=best_k, linkage='average')
    labels_sample = agg.fit_predict(X_sample)

    # Dendrogram from a smaller subset of the sample
    dendro_sample_size = min(50, X_sample.shape[0])
    dendro_idx_local = rng.choice(X_sample.shape[0], size=dendro_sample_size, replace=False)
    dendro_orig_idx = sample_idx[dendro_idx_local]
    X_dendro = X_sample[dendro_idx_local]

    Z = linkage(X_dendro, method='average', metric='euclidean')
    plt.figure(figsize=(16, 6))
    dendrogram(
        Z,
        labels=[str(i) for i in dendro_orig_idx],
        leaf_rotation=90,
        leaf_font_size=6,
    )
    plt.title(f'Agglomerative dendrogram ({pair_name})')
    plt.xlabel('Sample index (from combined_cleaned.csv)')
    plt.ylabel('Height')
    plt.tight_layout()

    dendro_path = f'agglomerative_dendrogram_{pair_name}.png'
    plt.savefig(dendro_path, dpi=200, bbox_inches='tight')
    plt.close()
    print('Saved:', dendro_path)

    # Nearest-centroid approximation to label full dataset
    centroids = np.vstack([
        X_sample[labels_sample == i].mean(axis=0)
        for i in range(best_k)
    ])
    clusters_full, _ = pairwise_distances_argmin_min(
        X_scaled,
        centroids,
        metric='euclidean',
    )

    out_ag = df[[x_col, y_col]].copy()
    out_ag[f'agglomerative_cluster_{pair_name}'] = clusters_full
    out_ag_path = f'combined_cleaned_{pair_name}_agglomerative.csv'
    out_ag.to_csv(out_ag_path, index=False)
    print('Agglomerative cluster counts:')
    print(out_ag[f'agglomerative_cluster_{pair_name}'].value_counts().sort_index())
    print('Saved:', out_ag_path)

    # Agglomerative scatter save
    plt.figure(figsize=(7, 5))
    plt.scatter(out_ag[x_col], out_ag[y_col], c=clusters_full, s=5, cmap='tab10', alpha=0.6)
    plt.title(f'Agglomerative clusters ({pair_name}), k={best_k}')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.savefig(f'agglomerative_scatter_{pair_name}.png', dpi=200, bbox_inches='tight')
    plt.close()

    # Merge K-Means + Agglomerative labels for this pair
    merged = df[[x_col, y_col]].copy()
    merged[f'kmeans_cluster_{pair_name}'] = clusters_kmeans
    merged[f'agglomerative_cluster_{pair_name}'] = clusters_full
    merged_path = f'combined_cleaned_{pair_name}_clusters_all.csv'
    merged.to_csv(merged_path, index=False)
    print('Saved:', merged_path)

    return best_k

In [4]:
# Pairwise clustering on the three combinations from your image
pairs = [
    ('bmi', 'glucose', 'bmi_glucose'),
    ('bmi', 'age', 'bmi_age'),
    ('glucose', 'age', 'glucose_age'),
]

best_ks = {}
for x_col, y_col, pair_name in pairs:
    best_ks[pair_name] = run_pairwise_clustering(data, x_col, y_col, pair_name)

print('\nSummary of best_k per pair:', best_ks)


=== bmi_glucose: saved combined_cleaned_bmi_glucose.csv ===

Best k by silhouette for bmi_glucose: 3


K-Means cluster counts:
kmeans_cluster_bmi_glucose
0    68727
1    51735
2    74343
Name: count, dtype: int64


Saved: agglomerative_dendrogram_bmi_glucose.png
Agglomerative cluster counts:
agglomerative_cluster_bmi_glucose
0    189539
1      2072
2      3194
Name: count, dtype: int64
Saved: combined_cleaned_bmi_glucose_agglomerative.csv


Saved: combined_cleaned_bmi_glucose_clusters_all.csv

=== bmi_age: saved combined_cleaned_bmi_age.csv ===


...

Saved: combined_cleaned_bmi_age_clusters_all.csv

=== glucose_age: saved combined_cleaned_glucose_age.csv ===


Best k by silhouette for glucose_age: 2


K-Means cluster counts:
kmeans_cluster_glucose_age
0    111961
1     82844
Name: count, dtype: int64


Saved: agglomerative_dendrogram_glucose_age.png
Agglomerative cluster counts:
agglomerative_cluster_glucose_age
0    190163
1      4642
Name: count, dtype: int64
Saved: combined_cleaned_glucose_age_agglomerative.csv


Saved: combined_cleaned_glucose_age_clusters_all.csv

Summary of best_k per pair: {'bmi_glucose': 3, 'bmi_age': 2, 'glucose_age': 2}
